In [1]:
import copy
from heapq import heappush, heappop

# Bạn có thể thay đổi N thành 4 (15-puzzle), 5 (24-puzzle), v.v.
N = 4

# Các hướng di chuyển của ô trống (0): Dưới, Trái, Trên, Phải
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class Node:
    def __init__(self, parent, mats, empty_tile_posi, g, h):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.g = g # Chi phí thực tế từ điểm bắt đầu (số bước)
        self.h = h # Chi phí heuristic ước lượng đến đích
        self.f = g + h # Tổng chi phí

    # Định nghĩa phép so sánh cho Priority Queue (ưu tiên f nhỏ nhất)
    def __lt__(self, nxt):
        return self.f < nxt.f

# Hàm Heuristic: Khoảng cách Manhattan (Tối ưu hơn rất nhiều so với đếm ô sai)
def calculateManhattan(mats, final):
    cost = 0
    for i in range(N):
        for j in range(N):
            val = mats[i][j]
            if val != 0:
                # Tìm vị trí của giá trị 'val' trong ma trận đích
                for x in range(N):
                    for y in range(N):
                        if final[x][y] == val:
                            cost += abs(x - i) + abs(y - j)
    return cost

def isSafe(x, y):
    return 0 <= x < N and 0 <= y < N

def printMatrix(mats):
    for i in range(N):
        for j in range(N):
            print(f"{mats[i][j]:2d}", end=" ")
        print()
    print("-" * 15)

def printPath(root):
    if root is None:
        return
    printPath(root.parent)
    printMatrix(root.mats)

def solve_npuzzle(initial, empty_tile_posi, final):
    pq = []

    # Tính chi phí Heuristic ban đầu
    h = calculateManhattan(initial, final)
    root = Node(None, initial, empty_tile_posi, 0, h)

    heappush(pq, root)

    # Dùng set để lưu các trạng thái đã duyệt (tránh lặp vòng)
    visited = set()

    while pq:
        minimum = heappop(pq)

        # Nếu h = 0 nghĩa là đã đạt trạng thái đích
        if minimum.h == 0:
            print("Đã tìm thấy đường đi! Các bước di chuyển:")
            printPath(minimum)
            print(f"Tổng số bước (g): {minimum.g}")
            return

        # Lưu trạng thái hiện tại dưới dạng tuple để đưa vào set (có thể hash)
        state_tuple = tuple(tuple(row) for row in minimum.mats)
        if state_tuple in visited:
            continue
        visited.add(state_tuple)

        # Sinh các trạng thái con
        for i in range(4):
            new_x = minimum.empty_tile_posi[0] + rows[i]
            new_y = minimum.empty_tile_posi[1] + cols[i]

            if isSafe(new_x, new_y):
                # Tạo ma trận mới
                new_mats = copy.deepcopy(minimum.mats)

                # Hoán đổi vị trí ô trống
                curr_x, curr_y = minimum.empty_tile_posi
                new_mats[curr_x][curr_y], new_mats[new_x][new_y] = new_mats[new_x][new_y], new_mats[curr_x][curr_y]

                new_state_tuple = tuple(tuple(row) for row in new_mats)
                if new_state_tuple not in visited:
                    h_new = calculateManhattan(new_mats, final)
                    child = Node(minimum, new_mats, [new_x, new_y], minimum.g + 1, h_new)
                    heappush(pq, child)

# --- KHỞI TẠO BÀI TOÁN 15-PUZZLE (N=4) ---
initial_4x4 = [
    [1, 2, 3, 4],
    [5, 6, 0, 8],
    [9, 10, 7, 11],
    [13, 14, 15, 12]
]

final_4x4 = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 0]
]

empty_pos = [1, 2] # Vị trí của ô số 0 trong bảng initial_4x4 (Hàng 1, Cột 2)
solve_npuzzle(initial_4x4, empty_pos, final_4x4)

Đã tìm thấy đường đi! Các bước di chuyển:
 1  2  3  4 
 5  6  0  8 
 9 10  7 11 
13 14 15 12 
---------------
 1  2  3  4 
 5  6  7  8 
 9 10  0 11 
13 14 15 12 
---------------
 1  2  3  4 
 5  6  7  8 
 9 10 11  0 
13 14 15 12 
---------------
 1  2  3  4 
 5  6  7  8 
 9 10 11 12 
13 14 15  0 
---------------
Tổng số bước (g): 3
